### Cell 1 — imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Cell 2 — create fact_orders table

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS mia_catalog.gold.fact_orders (
        cart_id INT,
        product_sk INT,
        customer_sk INT,
        date_sk INT,
        product_key STRING,
        customer_key STRING,
        order_date DATE,
        quantity INT,
        unit_price DOUBLE,
        line_total DOUBLE,
        discount_percentage DOUBLE,
        line_discounted_total DOUBLE,
        dw_created_at TIMESTAMP
    )
    USING DELTA
""")
print("mia_catalog.gold.fact_orders ready")

mia_catalog.gold.fact_orders ready


### Cell 3 — rebuild the product_id → sku lookup

In [0]:
product_id_schema = StructType([
    StructField("id", IntegerType()),
    StructField("sku", StringType()),
])

product_id_to_sku = (
    spark.table("bronze_dummyjson_products")
    .withColumn("parsed", from_json(col("raw_payload"), product_id_schema))
    .select(
        col("parsed.id").alias("dummyjson_product_id"),
        col("parsed.sku").alias("product_key"),
    )
)

print(f"Lookup rows: {product_id_to_sku.count()}")
display(product_id_to_sku.limit(5))

Lookup rows: 194


dummyjson_product_id,product_key
1,BEA-ESS-ESS-001
2,BEA-GLA-EYE-002
3,BEA-VEL-POW-003
4,BEA-CHI-LIP-004
5,BEA-NAI-NAI-005


### Cell 4 — translate product ids and derive order_date

In [0]:
order_lines_with_key = (
    spark.table("mia_catalog.silver.silver_order_lines").alias("ol")
    .join(
        product_id_to_sku.alias("lkp"),
        col("ol.product_id_ref") == col("lkp.dummyjson_product_id"),
        how="left"
    )
    .withColumn("order_date", to_date(col("ol.last_updated_ts")))
    .select(
        col("ol.cart_id"),
        col("ol.customer_key"),
        col("lkp.product_key"),
        col("order_date"),
        col("ol.quantity"),
        col("ol.unit_price"),
        col("ol.line_total"),
        col("ol.discount_percentage"),
        col("ol.line_discounted_total"),
    )
)

unmatched = order_lines_with_key.filter(col("product_key").isNull()).count()
total = order_lines_with_key.count()
print(f"Total order lines: {total}, unmatched product lookups: {unmatched}")

Total order lines: 800, unmatched product lookups: 0


### Cell 5 — as-of join to dim_date

In [0]:
dim_date_lookup = spark.table("mia_catalog.gold.dim_date").select("date_sk", "full_date")

with_date_sk = (
    order_lines_with_key.alias("ol")
    .join(dim_date_lookup.alias("dd"), col("ol.order_date") == col("dd.full_date"), how="left")
)

### Cell 6 — as-of join to dim_product

In [0]:
dim_product_lookup = (
    spark.table("mia_catalog.gold.dim_product")
    .select(
        col("product_sk"),
        col("product_key").alias("dp_product_key"),
        col("effective_start_date").alias("dp_effective_start_date"),
        col("effective_end_date").alias("dp_effective_end_date"),
    )
)

with_product_sk = (
    with_date_sk
    .join(
        dim_product_lookup,
        (col("product_key") == col("dp_product_key")) &
        (col("order_date") >= col("dp_effective_start_date")) &
        (col("order_date") < col("dp_effective_end_date")),
        how="left"
    )
    .drop("dp_product_key", "dp_effective_start_date", "dp_effective_end_date")
)

unmatched_product_sk = with_product_sk.filter(col("product_sk").isNull()).count()
print(f"Order lines with no matching product_sk: {unmatched_product_sk}")

Order lines with no matching product_sk: 0


### Cell 7 — as-of join to dim_customer

In [0]:
dim_customer_lookup = (
    spark.table("mia_catalog.gold.dim_customer")
    .select(
        col("customer_sk"),
        col("customer_key").alias("dc_customer_key"),
        col("effective_start_date").alias("dc_effective_start_date"),
        col("effective_end_date").alias("dc_effective_end_date"),
    )
)

with_customer_sk = (
    with_product_sk
    .join(
        dim_customer_lookup,
        (col("customer_key") == col("dc_customer_key")) &
        (col("order_date") >= col("dc_effective_start_date")) &
        (col("order_date") < col("dc_effective_end_date")),
        how="left"
    )
    .drop("dc_customer_key", "dc_effective_start_date", "dc_effective_end_date")
)

unmatched_customer_sk = with_customer_sk.filter(col("customer_sk").isNull()).count()
print(f"Order lines with no matching customer_sk: {unmatched_customer_sk}")

Order lines with no matching customer_sk: 0


### Cell 8 — final select and write

In [0]:
fact_orders_final = (
    with_customer_sk
    .select(
        "cart_id", "product_sk", "customer_sk", "date_sk",
        "product_key", "customer_key", "order_date",
        "quantity", "unit_price", "line_total",
        "discount_percentage", "line_discounted_total",
    )
    .withColumn("dw_created_at", current_timestamp())
)

fact_orders_final.write.format("delta").mode("overwrite").saveAsTable("mia_catalog.gold.fact_orders")
print(f"fact_orders rebuilt: {fact_orders_final.count()} rows")

fact_orders rebuilt: 800 rows


### Cell 9 — verify

In [0]:
display(spark.sql("""
    SELECT count(*) as total_fact_rows,
           count(product_sk) as matched_product_sk,
           count(customer_sk) as matched_customer_sk,
           count(date_sk) as matched_date_sk
    FROM mia_catalog.gold.fact_orders
"""))

display(spark.sql("""
    SELECT f.cart_id, f.order_date, f.unit_price as line_price_paid,
           dp.price as dim_product_price_at_that_version, dp.is_current
    FROM mia_catalog.gold.fact_orders f
    JOIN mia_catalog.gold.dim_product dp ON f.product_sk = dp.product_sk
    ORDER BY f.order_date DESC
    LIMIT 10
"""))

total_fact_rows,matched_product_sk,matched_customer_sk,matched_date_sk
800,800,800,800


cart_id,order_date,line_price_paid,dim_product_price_at_that_version,is_current
151,2026-08-12,19.99,22.988499999999995,true
5,2026-08-12,14.99,17.2385,true
178,2026-08-12,12.99,14.9385,true
142,2026-08-12,8.99,10.3385,true
163,2026-08-12,49.99,57.488499999999995,true
9,2026-08-12,9.99,11.4885,true
7,2026-08-12,69.99,80.48849999999999,true
4,2026-08-12,89.99,103.48849999999999,true
191,2026-08-12,129.99,149.4885,true
187,2026-08-12,79.99,91.98849999999999,true


In [0]:
display(spark.sql("""
    SELECT MIN(effective_start_date) as earliest_dim_product_date
    FROM mia_catalog.gold.dim_product
"""))
display(spark.sql("""
    SELECT MIN(order_date) as earliest_order_date
    FROM (SELECT to_date(last_updated_ts) as order_date FROM mia_catalog.silver.silver_order_lines)
"""))

earliest_dim_product_date
1900-01-01


earliest_order_date
2026-08-12


### Backdate dim_product's first versions:

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

first_versions = (
    spark.table("mia_catalog.gold.dim_product")
    .withColumn("rn", row_number().over(Window.partitionBy("product_key").orderBy("effective_start_date")))
    .filter(col("rn") == 1)
    .select("product_sk")
)

first_version_sks = [row["product_sk"] for row in first_versions.collect()]
sks_sql = ", ".join([str(sk) for sk in first_version_sks])

spark.sql(f"""
    UPDATE mia_catalog.gold.dim_product
    SET effective_start_date = DATE('1900-01-01')
    WHERE product_sk IN ({sks_sql})
""")
print(f"Backdated {len(first_version_sks)} dim_product first-version rows to 1900-01-01")

Backdated 1195 dim_product first-version rows to 1900-01-01


### Backdate dim_customer's first versions (same pattern):

In [0]:
first_customer_versions = (
    spark.table("mia_catalog.gold.dim_customer")
    .withColumn("rn", row_number().over(Window.partitionBy("customer_key").orderBy("effective_start_date")))
    .filter(col("rn") == 1)
    .select("customer_sk")
)

first_customer_sks = [row["customer_sk"] for row in first_customer_versions.collect()]
sks_sql_c = ", ".join([str(sk) for sk in first_customer_sks])

spark.sql(f"""
    UPDATE mia_catalog.gold.dim_customer
    SET effective_start_date = DATE('1900-01-01')
    WHERE customer_sk IN ({sks_sql_c})
""")
print(f"Backdated {len(first_customer_sks)} dim_customer first-version rows to 1900-01-01")

Backdated 208 dim_customer first-version rows to 1900-01-01


In [0]:
display(spark.sql("SELECT MIN(effective_start_date) FROM mia_catalog.gold.dim_product"))
display(spark.sql("SELECT MIN(effective_start_date) FROM mia_catalog.gold.dim_customer"))

MIN(effective_start_date)
1900-01-01


MIN(effective_start_date)
1900-01-01
